# 🐜 Otimização por Colônia de Formigas

**Disciplina:** Inteligência Artificial  
**Curso:** Ciência da Computação  
**Objetivo desta aula:** Compreender o algoritmo ACO (Ant Colony Optimization), sua inspiração biológica no comportamento de formigas reais, o mecanismo de feromônio, as variantes do algoritmo e suas aplicações em problemas de otimização combinatória.

---

## 1. O que é Otimização por Colônia de Formigas?

A **Otimização por Colônia de Formigas** (ACO — *Ant Colony Optimization*) é uma meta-heurística probabilística proposta por **Marco Dorigo em 1992**, inspirada no comportamento de forrageamento das formigas reais. Formigas reais são capazes de encontrar o caminho mais curto entre o formigueiro e uma fonte de alimento utilizando comunicação indireta por meio de substâncias químicas chamadas **feromônios**.

### Características Fundamentais

- **Inspiração biológica:** modela o comportamento coletivo de formigas que depositam e seguem trilhas de feromônio.
- **Comunicação estigmérgica:** as formigas se comunicam indiretamente através de modificações no ambiente (feromônio), sem comunicação direta entre indivíduos.
- **Aprendizado positivo e negativo:** caminhos bons recebem mais feromônio (reforço positivo); o feromônio evapora ao longo do tempo (esquecimento).
- **Otimização combinatória:** especialmente eficaz para problemas em grafos (TSP, roteamento, escalonamento).
- **Decisão probabilística:** cada formiga constrói soluções escolhendo arestas probabilisticamente baseada em feromônio e heurística.
- **Emergência coletiva:** soluções de alta qualidade emergem da interação de muitas formigas simples.

### O Comportamento Real das Formigas

Experimentos de Goss et al. (1989) demonstraram que formigas *Linepithema humile* colocadas em uma ponte de dois caminhos (um mais curto, outro mais longo) convergem naturalmente para o caminho mais curto:

1. Inicialmente, as formigas escolhem ambos os caminhos aleatoriamente.
2. As formigas no caminho mais curto chegam mais rápido e voltam mais cedo — depositando mais feromônio por unidade de tempo.
3. Formigas futuras são atraídas probabilisticamente pelo caminho com mais feromônio.
4. O processo se auto-reforça até que quase todas as formigas usem o caminho mais curto.

### Analogia com Otimização

```
Problema real (TSP)           →  Abstração ACO
─────────────────────────────────────────────────
Formiga                       →  Agente construtor de solução
Trilha entre formigueiro      →  Aresta no grafo do problema
e fonte de alimento
Feromônio na trilha           →  τ(i,j): qualidade acumulada da aresta
Distância da trilha           →  η(i,j): heurística (1/distância)
Formiga encontra comida       →  Solução completa construída
Formiga retorna e deposita    →  Atualização do feromônio
feromônio
Evaporação do feromônio       →  Mecanismo de esquecimento/diversificação
```

## 2. Breve História

| Ano | Evento |
|-----|--------|
| **1959** | Pierre-Paul Grassé descreve a **estigmergia** — comunicação indireta de insetos sociais através de modificações no ambiente. |
| **1989** | Goss et al. publicam o experimento da **ponte dupla** com formigas *Linepithema humile*, demonstrando convergência para o caminho mais curto. |
| **1992** | **Marco Dorigo** propõe o **Ant System (AS)** em sua tese de doutorado na Politécnica de Milão — o primeiro algoritmo ACO, aplicado ao TSP. |
| **1996** | Dorigo, Maniezzo e Colorni publicam **"Ant System: Optimization by a Colony of Cooperating Agents"** no *IEEE Transactions on Systems, Man, and Cybernetics*. |
| **1997** | Stutzle & Dorigo propõem o **ACS** (Ant Colony System), com regra de atualização de feromônio local e regra de decisão pseudo-aleatória proporcional. |
| **1997** | Stutzle & Hoos propõem o **MAX-MIN Ant System (MMAS)**, limitando os valores de feromônio a um intervalo $[\tau_{min}, \tau_{max}]$ para evitar estagnação. |
| **1999** | Dorigo & Di Caro generalizam o framework como **ACO** e propõem aplicações além do TSP: roteamento em redes, escalonamento, coloração de grafos. |
| **2002** | Dorigo & Stützle publicam o livro **"Ant Colony Optimization"** (MIT Press), consolidando o campo. |
| **2004** | Primeira aplicação bem-sucedida de ACO a problemas contínuos de otimização com **ACOR** (ACO for Continuous domains). |
| **Atualidade** | ACO é amplamente utilizado em logística, telecomunicações, bioinformática (sequenciamento de proteínas), robótica de enxame e otimização de rotas. |

## 3. Conceitos Fundamentais

### Componentes do ACO

| Símbolo | Nome | Descrição |
|---------|------|-----------|
| $\tau_{ij}$ | **Feromônio** | Quantidade de feromônio na aresta $(i,j)$. Representa a "memória coletiva" sobre a qualidade desse caminho. |
| $\eta_{ij}$ | **Heurística** | Informação a priori sobre a qualidade da aresta $(i,j)$. No TSP: $\eta_{ij} = 1/d_{ij}$ (inverso da distância). |
| $\alpha$ | **Alfa** | Controla a influência do feromônio $\tau$ na decisão. |
| $\beta$ | **Beta** | Controla a influência da heurística $\eta$ na decisão. |
| $\rho$ | **Taxa de evaporação** | Fração de feromônio que evapora a cada iteração. $\rho \in (0,1)$. |
| $Q$ | **Constante de depósito** | Quantidade total de feromônio depositada por uma formiga (proporcional à qualidade da solução). |
| $m$ | **Número de formigas** | Quantidade de formigas na colônia. |

### Regra de Transição Probabilística

A probabilidade de uma formiga $k$ no nó $i$ escolher ir para o nó $j$ é:

$$p_{ij}^k = \frac{[\tau_{ij}]^\alpha \cdot [\eta_{ij}]^\beta}{\sum_{l \in N_i^k} [\tau_{il}]^\alpha \cdot [\eta_{il}]^\beta}$$

onde $N_i^k$ é o conjunto de nós ainda não visitados pela formiga $k$ (lista de candidatos).

### Atualização do Feromônio

**Evaporação** (acontece sempre):
$$\tau_{ij} \leftarrow (1 - \rho) \cdot \tau_{ij}$$

**Depósito** (após cada formiga completar seu tour):
$$\tau_{ij} \leftarrow \tau_{ij} + \sum_{k=1}^{m} \Delta\tau_{ij}^k$$

onde $\Delta\tau_{ij}^k = Q / L_k$ se a aresta $(i,j)$ pertence ao tour da formiga $k$, e $0$ caso contrário. $L_k$ é o comprimento do tour da formiga $k$.

### Parâmetros Típicos

| Parâmetro | Valor Típico | Efeito |
|-----------|-------------|--------|
| $\alpha = 1$ | $[1, 2]$ | $\alpha$ alto → favorece feromônio (explotação) |
| $\beta = 2$–$5$ | $[2, 5]$ | $\beta$ alto → favorece heurística (guloso) |
| $\rho = 0.1$–$0.5$ | $[0.1, 0.9]$ | $\rho$ alto → evaporação rápida (diversificação) |
| $m = n$ | $10$–$50$ | Mais formigas → mais exploração |
| $Q = 100$ | qualquer | Escala do depósito de feromônio |

## 4. O Algoritmo ACO — Ant System

### Pseudocódigo

```
ALGORITMO Ant System (AS):

1. INICIALIZAÇÃO:
   τ(i,j) ← τ₀   (para todas as arestas)
   η(i,j) ← 1/d(i,j)  (heurística de distância)

2. LOOP PRINCIPAL (até critério de parada):

   2a. CONSTRUÇÃO DE SOLUÇÕES:
       Para cada formiga k = 1..m:
         Posicionar formiga em nó inicial aleatório
         Enquanto a solução estiver incompleta:
           Calcular probabilidades p(i,j)^k para todos j não visitados
           Escolher próximo nó j por roleta (roulette wheel)
           Mover para j e marcá-lo como visitado
         Calcular comprimento L_k do tour construído

   2b. ATUALIZAÇÃO DO FEROMÔNIO:
       Evaporação: τ(i,j) ← (1-ρ) · τ(i,j)   para todas as arestas
       Depósito:   τ(i,j) ← τ(i,j) + Σ_k Δτ(i,j)^k
         onde Δτ(i,j)^k = Q/L_k se (i,j) ∈ tour_k, senão 0

   2c. REGISTRAR melhor solução encontrada até agora

3. RETORNAR melhor solução encontrada
```

### Variantes do ACO

| Variante | Proposta por | Diferença principal |
|----------|-------------|---------------------|
| **Ant System (AS)** | Dorigo, 1992 | Versão original: todas as formigas depositam feromônio |
| **Ant Colony System (ACS)** | Dorigo & Gambardella, 1997 | Regra pseudo-aleatória proporcional; atualização local do feromônio |
| **MAX-MIN Ant System (MMAS)** | Stutzle & Hoos, 1997 | Limita $\tau \in [\tau_{min}, \tau_{max}]$; apenas a melhor formiga deposita |
| **Rank-Based AS** | Bullnheimer et al., 1999 | Formigas depositam feromônio proporcionalmente ao rank da solução |
| **ACO$_R$** | Socha & Dorigo, 2004 | Versão para domínios contínuos usando mistura de gaussianas |
| **MOPACO** | Múltiplos autores | Extensão multiobjetivo com arquivo Pareto |

## 5. Implementação do Zero — ACO para o TSP

A seguir, implementamos o **Ant System** completo para resolver o **Problema do Caixeiro Viajante (TSP)** sem uso de bibliotecas externas de ACO.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

np.random.seed(42)
random.seed(42)

# ─────────────────────────────────────────────────────────────
# Instância do TSP: 20 cidades com coordenadas aleatórias
# ─────────────────────────────────────────────────────────────
N_CIDADES = 20
coordenadas = np.random.rand(N_CIDADES, 2) * 100  # cidades em espaço [0,100]^2

def calcular_matriz_distancias(coords):
    """Calcula a matriz de distâncias euclidianas entre cidades."""
    n = len(coords)
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                dx = coords[i][0] - coords[j][0]
                dy = coords[i][1] - coords[j][1]
                dist[i][j] = np.sqrt(dx**2 + dy**2)
    return dist

distancias = calcular_matriz_distancias(coordenadas)

def comprimento_tour(tour, dist):
    """Calcula o comprimento total de um tour."""
    total = 0.0
    n = len(tour)
    for i in range(n):
        total += dist[tour[i]][tour[(i + 1) % n]]
    return total

# ─────────────────────────────────────────────────────────────
# Implementação do Ant System
# ─────────────────────────────────────────────────────────────
class AntSystem:
    """
    Ant System (AS) para o Problema do Caixeiro Viajante.
    Implementação fiel ao algoritmo original de Dorigo (1992).
    """

    def __init__(self, distancias, n_formigas=20, alfa=1.0, beta=3.0,
                 rho=0.3, Q=100.0, max_iter=100, tau_inicial=None):
        self.dist = distancias
        self.n = len(distancias)
        self.n_formigas = n_formigas
        self.alfa = alfa          # influência do feromônio
        self.beta = beta          # influência da heurística
        self.rho = rho            # taxa de evaporação
        self.Q = Q                # constante de depósito
        self.max_iter = max_iter

        # Heurística: inverso das distâncias (evitamos divisão por zero na diagonal)
        with np.errstate(divide='ignore', invalid='ignore'):
            self.eta = np.where(distancias > 0, 1.0 / distancias, 0.0)

        # Inicialização do feromônio
        if tau_inicial is None:
            tau_inicial = 1.0 / (self.n * np.mean(distancias[distancias > 0]))
        self.tau = np.full((self.n, self.n), tau_inicial)
        np.fill_diagonal(self.tau, 0.0)

        self.melhor_tour = None
        self.melhor_custo = float('inf')
        self.historico_melhor = []
        self.historico_medio = []

    def _construir_tour(self):
        """Uma formiga constrói um tour completo probabilisticamente."""
        cidade_inicial = random.randint(0, self.n - 1)
        tour = [cidade_inicial]
        nao_visitados = set(range(self.n)) - {cidade_inicial}

        while nao_visitados:
            atual = tour[-1]
            # Calcular numeradores da regra de transição
            candidatos = list(nao_visitados)
            numeradores = np.array([
                (self.tau[atual][j] ** self.alfa) * (self.eta[atual][j] ** self.beta)
                for j in candidatos
            ])
            soma = numeradores.sum()
            if soma == 0:
                probs = np.ones(len(candidatos)) / len(candidatos)
            else:
                probs = numeradores / soma

            # Seleção por roleta
            proximo = candidatos[np.random.choice(len(candidatos), p=probs)]
            tour.append(proximo)
            nao_visitados.remove(proximo)

        return tour

    def _atualizar_feromonio(self, tours, custos):
        """Evaporação e depósito de feromônio."""
        # Evaporação
        self.tau *= (1.0 - self.rho)

        # Depósito por cada formiga
        for tour, custo in zip(tours, custos):
            delta = self.Q / custo
            for i in range(self.n):
                a, b = tour[i], tour[(i + 1) % self.n]
                self.tau[a][b] += delta
                self.tau[b][a] += delta  # grafo não-direcionado

    def otimizar(self, verbose=True):
        """Executa o Ant System por max_iter iterações."""
        for iteracao in range(self.max_iter):
            # Cada formiga constrói um tour
            tours = [self._construir_tour() for _ in range(self.n_formigas)]
            custos = [comprimento_tour(t, self.dist) for t in tours]

            # Atualizar melhor global
            for tour, custo in zip(tours, custos):
                if custo < self.melhor_custo:
                    self.melhor_custo = custo
                    self.melhor_tour = tour[:]

            self.historico_melhor.append(self.melhor_custo)
            self.historico_medio.append(np.mean(custos))

            # Atualizar feromônio
            self._atualizar_feromonio(tours, custos)

            if verbose and iteracao % 20 == 0:
                print(f"  Iter {iteracao:4d} | Melhor: {self.melhor_custo:.2f} | Médio: {np.mean(custos):.2f}")

        return self.melhor_tour, self.melhor_custo


# ─────────────────────────────────────────────────────────────
# Executar o Ant System
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print("  Ant System — TSP com", N_CIDADES, "cidades")
print("=" * 55)

np.random.seed(42)
random.seed(42)

aco = AntSystem(
    distancias=distancias,
    n_formigas=20,
    alfa=1.0,
    beta=3.0,
    rho=0.3,
    Q=100.0,
    max_iter=100
)

melhor_tour, melhor_custo = aco.otimizar(verbose=True)

print(f"\nMelhor tour encontrado: {melhor_tour}")
print(f"Comprimento do tour:    {melhor_custo:.4f}")

# ─────────────────────────────────────────────────────────────
# Visualização: tour e convergência
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Tour encontrado pelo ACO
ax = axes[0]
tour_plot = melhor_tour + [melhor_tour[0]]
xs = coordenadas[tour_plot, 0]
ys = coordenadas[tour_plot, 1]
ax.plot(xs, ys, 'b-o', markersize=8, linewidth=1.5)
for i, (x, y) in enumerate(coordenadas):
    ax.annotate(str(i), (x, y), textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_title(f'Tour ACO\nComprimento = {melhor_custo:.2f}')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.grid(True, alpha=0.3)

# 2. Convergência
ax = axes[1]
ax.plot(aco.historico_melhor, 'b-', linewidth=2, label='Melhor')
ax.plot(aco.historico_medio, 'r--', linewidth=1.5, alpha=0.7, label='Médio')
ax.set_title('Convergência do Ant System')
ax.set_xlabel('Iteração')
ax.set_ylabel('Comprimento do Tour')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Mapa de feromônio
ax = axes[2]
tau_norm = aco.tau / aco.tau.max()
for i in range(N_CIDADES):
    for j in range(i + 1, N_CIDADES):
        intensidade = tau_norm[i][j]
        if intensidade > 0.1:  # mostrar apenas arestas com feromônio significativo
            ax.plot([coordenadas[i][0], coordenadas[j][0]],
                    [coordenadas[i][1], coordenadas[j][1]],
                    'b-', alpha=float(intensidade), linewidth=float(intensidade) * 3)
ax.scatter(coordenadas[:, 0], coordenadas[:, 1], c='red', s=60, zorder=5)
ax.set_title('Mapa de Feromônio\n(intensidade = feromônio)')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.grid(True, alpha=0.3)

plt.suptitle('Ant System — Otimização por Colônia de Formigas', fontsize=13)
plt.tight_layout()
plt.show()
print("\n✅ Visualização gerada com sucesso.")

## 6. Análise dos Parâmetros do ACO

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Usar a mesma instância do TSP
np.random.seed(0)
coordenadas_exp = np.random.rand(15, 2) * 100
distancias_exp = calcular_matriz_distancias(coordenadas_exp)

def rodar_aco(distancias, alfa, beta, rho, n_rep=5, max_iter=80, n_formigas=15):
    """Executa ACO múltiplas vezes e retorna histórico médio."""
    todos_historicos = []
    for seed in range(n_rep):
        np.random.seed(seed * 7)
        random.seed(seed * 7)
        aco = AntSystem(distancias, n_formigas=n_formigas, alfa=alfa,
                        beta=beta, rho=rho, Q=100.0, max_iter=max_iter)
        aco.otimizar(verbose=False)
        todos_historicos.append(aco.historico_melhor)
    return np.mean(todos_historicos, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Efeito de alfa ---
ax = axes[0]
for alfa in [0.5, 1.0, 2.0, 4.0]:
    hist = rodar_aco(distancias_exp, alfa=alfa, beta=3.0, rho=0.3)
    ax.plot(hist, label=f'α={alfa}', linewidth=2)
ax.set_title('Efeito de α (influência do feromônio)\nβ=3, ρ=0.3')
ax.set_xlabel('Iteração')
ax.set_ylabel('Melhor comprimento')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Efeito de beta ---
ax = axes[1]
for beta in [1.0, 2.0, 4.0, 6.0]:
    hist = rodar_aco(distancias_exp, alfa=1.0, beta=beta, rho=0.3)
    ax.plot(hist, label=f'β={beta}', linewidth=2)
ax.set_title('Efeito de β (influência da heurística)\nα=1, ρ=0.3')
ax.set_xlabel('Iteração')
ax.set_ylabel('Melhor comprimento')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Efeito de rho ---
ax = axes[2]
for rho in [0.05, 0.2, 0.5, 0.9]:
    hist = rodar_aco(distancias_exp, alfa=1.0, beta=3.0, rho=rho)
    ax.plot(hist, label=f'ρ={rho}', linewidth=2)
ax.set_title('Efeito de ρ (taxa de evaporação)\nα=1, β=3')
ax.set_xlabel('Iteração')
ax.set_ylabel('Melhor comprimento')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Análise de Sensibilidade dos Parâmetros do ACO', fontsize=13)
plt.tight_layout()
plt.show()
print("✅ Análise de parâmetros concluída.")

## 7. ACO com Biblioteca — Implementação usando NetworkX + ACO personalizado

A seguir demonstramos como integrar o ACO com estruturas de dados do **NetworkX** para representar o grafo do problema, além de comparar com uma solução gulosa (*nearest neighbor heuristic*) como baseline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────
# Heurística Gulosa (Nearest Neighbor) como baseline
# ─────────────────────────────────────────────────────────────
def heuristica_vizinho_mais_proximo(dist, inicio=0):
    """Constrói um tour guloso sempre indo à cidade mais próxima não visitada."""
    n = len(dist)
    visitados = [False] * n
    tour = [inicio]
    visitados[inicio] = True
    for _ in range(n - 1):
        atual = tour[-1]
        melhor_j = -1
        melhor_d = float('inf')
        for j in range(n):
            if not visitados[j] and dist[atual][j] < melhor_d:
                melhor_d = dist[atual][j]
                melhor_j = j
        tour.append(melhor_j)
        visitados[melhor_j] = True
    return tour

# ─────────────────────────────────────────────────────────────
# Comparação: ACO vs Guloso em diferentes tamanhos de instância
# ─────────────────────────────────────────────────────────────
tamanhos = [10, 15, 20, 25, 30]
resultados_aco = []
resultados_guloso = []

print("Comparação ACO vs Heurística Gulosa")
print(f"{'N':>5} | {'ACO':>10} | {'Guloso':>10} | {'Melhoria':>10}")
print("-" * 45)

for n_cid in tamanhos:
    np.random.seed(123)
    coords = np.random.rand(n_cid, 2) * 100
    dist = calcular_matriz_distancias(coords)

    # ACO
    np.random.seed(42)
    random.seed(42)
    aco_inst = AntSystem(dist, n_formigas=n_cid, alfa=1.0, beta=3.0,
                         rho=0.3, Q=100.0, max_iter=80)
    _, custo_aco = aco_inst.otimizar(verbose=False)

    # Guloso
    tour_guloso = heuristica_vizinho_mais_proximo(dist, inicio=0)
    custo_guloso = comprimento_tour(tour_guloso, dist)

    melhoria = (custo_guloso - custo_aco) / custo_guloso * 100
    resultados_aco.append(custo_aco)
    resultados_guloso.append(custo_guloso)

    print(f"{n_cid:>5} | {custo_aco:>10.2f} | {custo_guloso:>10.2f} | {melhoria:>+9.1f}%")

# Gráfico de comparação
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x = np.arange(len(tamanhos))
largura = 0.35
ax.bar(x - largura/2, resultados_guloso, largura, label='Guloso (NN)', color='coral', alpha=0.8)
ax.bar(x + largura/2, resultados_aco, largura, label='ACO', color='steelblue', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'n={n}' for n in tamanhos])
ax.set_title('ACO vs Heurística Gulosa\n(menor = melhor)')
ax.set_ylabel('Comprimento do Tour')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

ax = axes[1]
melhoria_pct = [(g - a) / g * 100 for g, a in zip(resultados_guloso, resultados_aco)]
cores = ['green' if m > 0 else 'red' for m in melhoria_pct]
ax.bar([f'n={n}' for n in tamanhos], melhoria_pct, color=cores, alpha=0.8)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Melhoria do ACO sobre o Guloso (%)')
ax.set_ylabel('Redução do comprimento (%)')
ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('Comparação: ACO vs Heurística Gulosa no TSP', fontsize=13)
plt.tight_layout()
plt.show()
print("\n✅ Comparação concluída.")

## 8. Comparação: ACO vs. Outros Algoritmos

| Critério | ACO | Algoritmo Genético | PSO | Busca Tabu |
|----------|-----|-------------------|-----|------------|
| **Tipo de problema** | Combinatório (grafos) | Combinatório/Contínuo | Contínuo | Combinatório |
| **Representação de solução** | Construção incremental | String/permutação | Vetor real | Vizinhança |
| **Memória** | Feromônio coletivo | Sem memória individual | pbest/gbest | Lista tabu |
| **Comunicação** | Estigmérgia (indireta) | Crossover/mutação | Velocidade social | Nenhuma |
| **Paralelismo** | Formigas em paralelo | Indivíduos em paralelo | Partículas em paralelo | Sequencial |
| **Exploração/Explotação** | Feromônio + heurística | Crossover + mutação | Inércia + social | Lista tabu |
| **Facilidade de implementação** | Média | Média | Alta | Média |
| **Convergência** | Boa em grafos | Variável | Rápida (contínuo) | Boa (local) |
| **Sensibilidade a parâmetros** | Alta (α, β, ρ) | Alta | Média (w, c1, c2) | Média |

## 9. Aplicações do ACO

### Problemas de Roteamento
- **TSP (Caixeiro Viajante):** problema benchmark clássico do ACO
- **VRP (Vehicle Routing Problem):** otimização de frotas de entrega
- **Roteamento em redes:** protocolos adaptativos (AntNet) para Internet
- **Robótica:** planejamento de trajetórias e exploração de ambientes

### Problemas de Escalonamento
- **Job Shop Scheduling:** alocação de tarefas em máquinas
- **Escalonamento de projetos:** CPM/PERT com ACO
- **Alocação de frequências** em redes móveis

### Problemas de Otimização em Grafos
- **Coloração de grafos:** atribuição de cores com mínimo de conflitos
- **Cobertura de vértices:** NP-difícil, ACO com bons resultados
- **Árvore geradora mínima:** variante do ACO

### Bioinformática
- **Alinhamento de sequências** de DNA/proteínas
- **Predição de estrutura de proteínas**
- **Análise de redes biológicas**

## 10. Vantagens e Desvantagens

| Vantagens | Desvantagens |
|-----------|-------------|
| ✅ Excelente para problemas combinatórios em grafos | ❌ Convergência mais lenta que AG em alguns problemas |
| ✅ Adaptação dinâmica a mudanças no problema | ❌ Muitos parâmetros para ajustar (α, β, ρ, Q, m) |
| ✅ Paralelismo natural (formigas independentes) | ❌ Difícil de aplicar diretamente a domínios contínuos |
| ✅ Memória distribuída (feromônio no grafo) | ❌ Pode estagnar prematuramente sem diversificação |
| ✅ Incorpora conhecimento heurístico (η) | ❌ Overhead de manutenção da matriz de feromônio |
| ✅ Funciona bem em TSP e VRP em prática | ❌ Análise teórica de convergência complexa |

## 11. Exercícios Propostos

### Exercício 1 — Implementar MAX-MIN Ant System (MMAS)
Modifique a classe `AntSystem` para implementar o **MMAS** (Stützle & Hoos, 1997):
- Apenas a melhor formiga global deposita feromônio a cada iteração
- Os valores de feromônio são limitados ao intervalo $[\tau_{min}, \tau_{max}]$
- Reinicialização do feromônio quando estagnação é detectada

Compare com o Ant System padrão na instância de 20 cidades. Qual método encontra melhores soluções?

### Exercício 2 — ACO para o Problema da Mochila
Adapte o ACO para resolver o **Problema da Mochila 0/1** (*Knapsack Problem*):
- Cada formiga decide sequencialmente incluir ou não cada item
- Defina uma heurística adequada (ex: razão valor/peso)
- Respeite a restrição de capacidade na construção da solução

Teste com uma instância de 30 itens e compare com a solução ótima (programação dinâmica).

### Exercício 3 — Estigmergia Visual
Implemente uma **simulação visual** do comportamento emergente das formigas em um grid 2D:
- Grid 30×30 com um formigueiro e uma fonte de alimento
- Formigas se movem aleatoriamente até encontrar comida
- Ao retornar, depositam feromônio no caminho percorrido
- Visualize a evolução das trilhas de feromônio ao longo do tempo

### Exercício 4 — Comparação de Heurísticas
Para o TSP de 20 cidades, compare a qualidade de soluções do ACO usando três heurísticas $\eta_{ij}$ diferentes:
1. $\eta_{ij} = 1/d_{ij}$ (inverso da distância)
2. $\eta_{ij} = 1/d_{ij}^2$ (inverso do quadrado)
3. $\eta_{ij} = e^{-d_{ij}/\bar{d}}$ (exponencial negativa)

### Exercício 5 — ACO para Coloração de Grafos
Implemente o ACO para o problema de **coloração de grafos**: dado um grafo $G = (V, E)$, atribua cores aos vértices de modo que vértices adjacentes tenham cores diferentes, minimizando o número total de cores utilizadas.

## 12. Referências

- DORIGO, Marco; MANIEZZO, Vittorio; COLORNI, Alberto. Ant System: Optimization by a colony of cooperating agents. *IEEE Transactions on Systems, Man, and Cybernetics – Part B*, v. 26, n. 1, p. 29–41, 1996.

- DORIGO, Marco; GAMBARDELLA, Luca M. Ant Colony System: A cooperative learning approach to the Traveling Salesman Problem. *IEEE Transactions on Evolutionary Computation*, v. 1, n. 1, p. 53–66, 1997.

- STÜTZLE, Thomas; HOOS, Holger H. MAX–MIN Ant System. *Future Generation Computer Systems*, v. 16, n. 8, p. 889–914, 2000.

- DORIGO, Marco; STÜTZLE, Thomas. *Ant Colony Optimization*. Cambridge, MA: MIT Press, 2004.

- DORIGO, Marco; DI CARO, Gianni. Ant Colony Optimization: A new meta-heuristic. *Proceedings of the 1999 Congress on Evolutionary Computation*, v. 2, p. 1470–1477, 1999.

- SOCHA, Krzysztof; DORIGO, Marco. Ant colony optimization for continuous domains. *European Journal of Operational Research*, v. 185, n. 3, p. 1155–1173, 2008.

- GOSS, Simon et al. Self-organized shortcuts in the Argentine ant. *Naturwissenschaften*, v. 76, n. 12, p. 579–581, 1989.